# 03 - Performance Profiling & Batching

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the **tradeoffs** between batch size, throughput, and memory usage
- Find the **optimal batch size** for your hardware through systematic benchmarking
- Implement **gradient accumulation** to simulate large batch sizes on limited memory
- Optimize **DataLoader** performance with `num_workers`, `pin_memory`, and `prefetch_factor`
- Profile training steps using Python's `time` module to identify bottlenecks
- Apply these techniques to **optimize a training pipeline** end-to-end

## Prerequisites

- PyTorch fundamentals (tensors, `nn.Module`, training loops)
- Understanding of SGD and mini-batch training
- Basic knowledge of DataLoaders
- Previous notebooks: [01 - Experiment Tracking](01_Experiment_Tracking_Lightweight.ipynb), [02 - Model Saving & Loading](02_Model_Saving_Loading_Inference_Pipeline.ipynb)

## Table of Contents

1. [Batch Size Tradeoffs](#1-batch-size-tradeoffs)
2. [Benchmarking Different Batch Sizes](#2-benchmarking-different-batch-sizes)
3. [Finding the Optimal Batch Size](#3-finding-the-optimal-batch-size)
4. [Gradient Accumulation](#4-gradient-accumulation)
5. [DataLoader Optimization](#5-dataloader-optimization)
6. [CPU Profiling with the time Module](#6-cpu-profiling-with-the-time-module)
7. [Profiling a Complete Training Step](#7-profiling-a-complete-training-step)
8. [Exercise: Optimize a Training Pipeline](#8-exercise-optimize-a-training-pipeline)
9. [Common Mistakes & Debugging Tips](#9-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import os
import time
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

---
## 1. Batch Size Tradeoffs

Batch size is one of the most impactful hyperparameters, affecting training speed, memory usage, and model quality.

### Throughput vs Memory

| Batch Size | Throughput | Memory | Gradient Quality | Generalization |
|---|---|---|---|---|
| **Small** (8-32) | Lower (more overhead per sample) | Low | Noisy (acts as regularizer) | Often better |
| **Medium** (64-256) | Good balance | Moderate | Stable | Good |
| **Large** (512+) | Highest (full GPU utilization) | High (may OOM) | Very stable | May need tuning |

### Key relationships

- **Memory scales linearly** with batch size: doubling batch size roughly doubles memory
- **Throughput increases** with batch size up to a point, then plateaus (GPU saturated)
- **Steps per epoch decrease** with larger batches: `steps = ceil(N / batch_size)`
- **Wall-clock time per epoch** often has a sweet spot -- not the smallest or largest batch

### The linear scaling rule

When increasing batch size by factor `k`, increase learning rate by factor `k` to maintain similar training dynamics (Goyal et al., 2017).

In [ ]:
# Create a larger dataset for meaningful benchmarks
set_global_seed(42)

N_SAMPLES = 10000
N_FEATURES = 100
N_CLASSES = 10

X = torch.randn(N_SAMPLES, N_FEATURES)
true_w = torch.randn(N_FEATURES, N_CLASSES)
y = (X @ true_w).argmax(dim=1)

split = int(0.8 * N_SAMPLES)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print(f"Train: {X_train.shape[0]:,}, Val: {X_val.shape[0]:,}")
print(f"Features: {N_FEATURES}, Classes: {N_CLASSES}")

In [ ]:
class BenchmarkMLP(nn.Module):
    """Larger MLP for benchmarking (more parameters = more visible differences)."""
    def __init__(self, input_dim: int = 100, hidden_dim: int = 256,
                 output_dim: int = 10, n_layers: int = 4):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Count parameters
test_model = BenchmarkMLP()
n_params = sum(p.numel() for p in test_model.parameters())
print(f"Model parameters: {n_params:,}")

---
## 2. Benchmarking Different Batch Sizes

We measure two key metrics for each batch size:
- **Time per epoch** (wall-clock seconds)
- **Throughput** (samples processed per second)

In [ ]:
def benchmark_batch_size(
    batch_size: int,
    X_train: torch.Tensor,
    y_train: torch.Tensor,
    n_epochs: int = 3,
    device: torch.device = DEVICE,
) -> Dict[str, float]:
    """Benchmark training with a given batch size.

    Args:
        batch_size: Batch size to test.
        X_train, y_train: Training data.
        n_epochs: Number of epochs to average over.
        device: Device to run on.

    Returns:
        Dict with timing and throughput metrics.
    """
    set_global_seed(42)

    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
    )
    model = BenchmarkMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    n_steps = len(loader)
    epoch_times = []

    for epoch in range(n_epochs):
        model.train()
        t_start = time.perf_counter()

        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = loss_fn(out, y_batch)
            loss.backward()
            optimizer.step()

        epoch_time = time.perf_counter() - t_start
        epoch_times.append(epoch_time)

    avg_epoch_time = np.mean(epoch_times)
    throughput = len(X_train) / avg_epoch_time

    return {
        "batch_size": batch_size,
        "steps_per_epoch": n_steps,
        "avg_epoch_time": avg_epoch_time,
        "throughput_samples_per_sec": throughput,
        "epoch_times": epoch_times,
    }


# Benchmark a range of batch sizes
batch_sizes = [8, 16, 32, 64, 128, 256, 512, 1024]
benchmark_results = []

print(f"{'Batch Size':>10} {'Steps/Epoch':>12} {'Epoch Time (s)':>15} {'Throughput':>15}")
print("-" * 55)

for bs in batch_sizes:
    result = benchmark_batch_size(bs, X_train, y_train, n_epochs=3)
    benchmark_results.append(result)
    print(f"{result['batch_size']:>10} {result['steps_per_epoch']:>12} "
          f"{result['avg_epoch_time']:>15.4f} "
          f"{result['throughput_samples_per_sec']:>15,.0f}")

In [ ]:
# Visualize the tradeoffs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bs_list = [r["batch_size"] for r in benchmark_results]
epoch_times = [r["avg_epoch_time"] for r in benchmark_results]
throughputs = [r["throughput_samples_per_sec"] for r in benchmark_results]

# Time per epoch
axes[0].plot(bs_list, epoch_times, "o-", color="steelblue", linewidth=2)
axes[0].set_xlabel("Batch Size")
axes[0].set_ylabel("Time per Epoch (seconds)")
axes[0].set_title("Epoch Time vs Batch Size")
axes[0].set_xscale("log", base=2)
axes[0].grid(True, alpha=0.3)

# Throughput
axes[1].plot(bs_list, throughputs, "o-", color="coral", linewidth=2)
axes[1].set_xlabel("Batch Size")
axes[1].set_ylabel("Samples per Second")
axes[1].set_title("Throughput vs Batch Size")
axes[1].set_xscale("log", base=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identify best
best_throughput_idx = int(np.argmax(throughputs))
best_time_idx = int(np.argmin(epoch_times))
print(f"Highest throughput: batch_size={bs_list[best_throughput_idx]} "
      f"({throughputs[best_throughput_idx]:,.0f} samples/sec)")
print(f"Fastest epoch: batch_size={bs_list[best_time_idx]} "
      f"({epoch_times[best_time_idx]:.4f}s)")

---
## 3. Finding the Optimal Batch Size

Throughput alone is not enough. We also need to check that training **converges well** at the chosen batch size. Larger batches may train faster per epoch but converge to worse solutions.

In [ ]:
def train_and_evaluate(
    batch_size: int,
    lr: float,
    n_epochs: int = 20,
    device: torch.device = DEVICE,
) -> Dict[str, Any]:
    """Train for n_epochs and return final validation accuracy + timing."""
    set_global_seed(42)

    train_loader = DataLoader(
        TensorDataset(X_train.to(device), y_train.to(device)),
        batch_size=batch_size, shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(X_val.to(device), y_val.to(device)),
        batch_size=512,
    )

    model = BenchmarkMLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    t_start = time.perf_counter()
    train_losses = []
    val_accs = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            out = model(Xb)
            loss = loss_fn(out, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        train_losses.append(epoch_loss / n_batches)

        # Validate
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                preds = model(Xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        val_accs.append(correct / total)

    total_time = time.perf_counter() - t_start

    return {
        "batch_size": batch_size,
        "lr": lr,
        "final_val_acc": val_accs[-1],
        "best_val_acc": max(val_accs),
        "total_time": total_time,
        "train_losses": train_losses,
        "val_accs": val_accs,
    }


# Test batch sizes with linear scaling rule for LR
base_lr = 0.001
base_bs = 32

test_configs = []
for bs in [16, 32, 64, 128, 256, 512]:
    # Linear scaling: lr = base_lr * (bs / base_bs)
    scaled_lr = base_lr * (bs / base_bs)
    # Cap the LR to avoid instability
    scaled_lr = min(scaled_lr, 0.01)
    test_configs.append((bs, scaled_lr))

convergence_results = []
print(f"{'Batch Size':>10} {'LR':>10} {'Best Val Acc':>13} {'Total Time':>11}")
print("-" * 48)

for bs, lr in test_configs:
    result = train_and_evaluate(bs, lr, n_epochs=20)
    convergence_results.append(result)
    print(f"{bs:>10} {lr:>10.5f} {result['best_val_acc']:>13.4f} {result['total_time']:>11.2f}s")

In [ ]:
# Plot convergence curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for result in convergence_results:
    label = f"bs={result['batch_size']}"
    epochs = range(1, len(result["train_losses"]) + 1)
    ax1.plot(epochs, result["train_losses"], label=label, marker=".", markersize=3)
    ax2.plot(epochs, result["val_accs"], label=label, marker=".", markersize=3)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss")
ax1.set_title("Training Loss by Batch Size")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Accuracy")
ax2.set_title("Validation Accuracy by Batch Size")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Gradient Accumulation

**Problem:** You want the training dynamics of batch_size=256 but your GPU only fits batch_size=64.

**Solution:** Gradient accumulation. Run multiple small forward/backward passes, accumulate gradients, then do one optimizer step.

```python
# Effective batch size = batch_size * accum_steps
for i, (X_batch, y_batch) in enumerate(loader):
    loss = loss_fn(model(X_batch), y_batch)
    loss = loss / accum_steps       # Scale loss
    loss.backward()                  # Accumulate gradients
    if (i + 1) % accum_steps == 0:   # Step every accum_steps
        optimizer.step()
        optimizer.zero_grad()
```

### Why divide the loss?

- `loss.backward()` **adds** gradients to `.grad` (PyTorch default)
- After `accum_steps` backward passes, the accumulated gradient is `accum_steps` times larger
- Dividing by `accum_steps` makes the accumulated gradient equivalent to one large batch

In [ ]:
def train_with_gradient_accumulation(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    accum_steps: int,
    n_epochs: int = 10,
    device: torch.device = DEVICE,
) -> Dict[str, List[float]]:
    """Train with gradient accumulation.

    Args:
        model: The model.
        train_loader: Training DataLoader.
        val_loader: Validation DataLoader.
        optimizer: Optimizer.
        loss_fn: Loss function.
        accum_steps: Number of mini-batches to accumulate before stepping.
        n_epochs: Number of training epochs.
        device: Device.

    Returns:
        History dict with train_loss and val_acc per epoch.
    """
    history = {"train_loss": [], "val_acc": []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss = 0.0
        n_optimizer_steps = 0
        optimizer.zero_grad()  # Clear gradients at start

        for i, (Xb, yb) in enumerate(train_loader):
            Xb, yb = Xb.to(device), yb.to(device)
            out = model(Xb)
            loss = loss_fn(out, yb)
            loss = loss / accum_steps  # Scale loss by accumulation steps
            loss.backward()            # Gradients accumulate in .grad

            total_loss += loss.item() * accum_steps  # Track unscaled loss

            if (i + 1) % accum_steps == 0:
                optimizer.step()
                optimizer.zero_grad()
                n_optimizer_steps += 1

        # Handle leftover batches (if dataset not evenly divisible)
        if (i + 1) % accum_steps != 0:
            optimizer.step()
            optimizer.zero_grad()
            n_optimizer_steps += 1

        avg_loss = total_loss / (i + 1)
        history["train_loss"].append(avg_loss)

        # Validate
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                preds = model(Xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total
        history["val_acc"].append(val_acc)

        if epoch % 5 == 0 or epoch == 1:
            eff_bs = train_loader.batch_size * accum_steps
            print(f"Epoch {epoch:2d} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} "
                  f"| Optimizer steps: {n_optimizer_steps} (eff. bs={eff_bs})")

    return history

In [ ]:
# Compare: standard training vs gradient accumulation
# Target effective batch size: 256

# Method 1: Actual batch_size=256
print("=" * 60)
print("Method 1: Actual batch_size=256")
print("=" * 60)
set_global_seed(42)
model_large = BenchmarkMLP().to(DEVICE)
opt_large = torch.optim.Adam(model_large.parameters(), lr=0.001)
large_loader = DataLoader(
    TensorDataset(X_train.to(DEVICE), y_train.to(DEVICE)),
    batch_size=256, shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_val.to(DEVICE), y_val.to(DEVICE)),
    batch_size=512,
)
history_large = train_with_gradient_accumulation(
    model_large, large_loader, val_loader, opt_large, nn.CrossEntropyLoss(),
    accum_steps=1, n_epochs=20,
)

# Method 2: batch_size=64 with accum_steps=4 (effective batch=256)
print(f"\n{'=' * 60}")
print("Method 2: batch_size=64, accum_steps=4 (effective=256)")
print("=" * 60)
set_global_seed(42)
model_accum = BenchmarkMLP().to(DEVICE)
opt_accum = torch.optim.Adam(model_accum.parameters(), lr=0.001)
small_loader = DataLoader(
    TensorDataset(X_train.to(DEVICE), y_train.to(DEVICE)),
    batch_size=64, shuffle=True,
)
history_accum = train_with_gradient_accumulation(
    model_accum, small_loader, val_loader, opt_accum, nn.CrossEntropyLoss(),
    accum_steps=4, n_epochs=20,
)

In [ ]:
# Compare convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, 21)
ax1.plot(epochs, history_large["train_loss"], "o-", label="bs=256 (actual)", markersize=3)
ax1.plot(epochs, history_accum["train_loss"], "s--", label="bs=64, accum=4 (eff=256)", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss")
ax1.set_title("Training Loss: Actual vs Accumulated")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history_large["val_acc"], "o-", label="bs=256 (actual)", markersize=3)
ax2.plot(epochs, history_accum["val_acc"], "s--", label="bs=64, accum=4 (eff=256)", markersize=3)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Accuracy")
ax2.set_title("Validation Accuracy: Actual vs Accumulated")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final val acc (actual bs=256):     {history_large['val_acc'][-1]:.4f}")
print(f"Final val acc (accum eff bs=256):  {history_accum['val_acc'][-1]:.4f}")

---
## 5. DataLoader Optimization

The `DataLoader` can become a bottleneck if it cannot feed data to the GPU fast enough.

### Key parameters

| Parameter | What it does | Default | Recommended |
|---|---|---|---|
| `num_workers` | Parallel data loading processes | 0 (main process) | 2-8 (depends on CPU cores) |
| `pin_memory` | Pre-allocate page-locked memory for faster CPU-to-GPU transfer | `False` | `True` when using GPU |
| `prefetch_factor` | Number of batches each worker pre-loads | 2 | 2-4 |
| `persistent_workers` | Keep workers alive between epochs | `False` | `True` if `num_workers > 0` |

**Note:** On Windows, `num_workers > 0` requires the `if __name__ == '__main__':` guard in scripts. In Jupyter notebooks, `num_workers=0` is often safest on Windows.

In [ ]:
def benchmark_dataloader(
    num_workers: int,
    pin_memory: bool,
    batch_size: int = 64,
    n_iterations: int = 100,
) -> float:
    """Benchmark DataLoader iteration speed.

    Returns:
        Average time per batch in milliseconds.
    """
    loader_kwargs = {
        "batch_size": batch_size,
        "shuffle": True,
        "num_workers": num_workers,
        "pin_memory": pin_memory,
    }
    # Only add prefetch_factor and persistent_workers when num_workers > 0
    if num_workers > 0:
        loader_kwargs["prefetch_factor"] = 2
        loader_kwargs["persistent_workers"] = True

    loader = DataLoader(TensorDataset(X_train, y_train), **loader_kwargs)

    # Warm up
    for i, _ in enumerate(loader):
        if i >= 5:
            break

    # Benchmark
    times = []
    batch_count = 0
    for _ in range(3):  # Multiple passes
        for Xb, yb in loader:
            t0 = time.perf_counter()
            if pin_memory and DEVICE.type == "cuda":
                Xb = Xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
            else:
                Xb = Xb.to(DEVICE)
                yb = yb.to(DEVICE)
            times.append(time.perf_counter() - t0)
            batch_count += 1
            if batch_count >= n_iterations:
                break
        if batch_count >= n_iterations:
            break

    return np.mean(times) * 1000  # milliseconds


# Test different configurations
# Note: On Windows/macOS, num_workers > 0 in notebooks may cause issues.
# We test num_workers=0 with different pin_memory settings.
configs = [
    {"num_workers": 0, "pin_memory": False},
    {"num_workers": 0, "pin_memory": True},
]

# Optionally test multi-worker on Linux
import platform
if platform.system() == "Linux":
    configs.extend([
        {"num_workers": 2, "pin_memory": True},
        {"num_workers": 4, "pin_memory": True},
    ])

print(f"{'num_workers':>12} {'pin_memory':>12} {'ms/batch':>10}")
print("-" * 36)

for cfg in configs:
    avg_ms = benchmark_dataloader(**cfg)
    print(f"{cfg['num_workers']:>12} {str(cfg['pin_memory']):>12} {avg_ms:>10.3f}")

### DataLoader best practices summary

- **Always** set `pin_memory=True` when training on GPU
- Use `num_workers` equal to 2-4 times the number of GPUs (experiment to find optimum)
- Set `persistent_workers=True` to avoid worker startup overhead between epochs
- On Windows, `num_workers=0` in Jupyter is safest; use `num_workers > 0` in standalone scripts with the `if __name__ == '__main__':` guard
- Use `non_blocking=True` in `.to(device)` when using `pin_memory=True`

---
## 6. CPU Profiling with the time Module

Before reaching for complex profilers, Python's built-in `time` module can reveal major bottlenecks.

### Key functions

| Function | Resolution | Use case |
|---|---|---|
| `time.perf_counter()` | Highest available | General timing (recommended) |
| `time.time()` | ~1ms | Wall-clock timestamps |
| `time.process_time()` | ~1ms | CPU time only (excludes sleep/IO) |

In [ ]:
class SimpleTimer:
    """Context manager for timing code blocks."""

    def __init__(self, name: str = ""):
        self.name = name
        self.elapsed = 0.0

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start

    def __repr__(self):
        return f"{self.name}: {self.elapsed*1000:.2f}ms"


class StepProfiler:
    """Accumulate timing stats across multiple training steps."""

    def __init__(self):
        self.timings: Dict[str, List[float]] = {}

    def record(self, name: str, elapsed: float) -> None:
        if name not in self.timings:
            self.timings[name] = []
        self.timings[name].append(elapsed)

    def summary(self) -> Dict[str, Dict[str, float]]:
        result = {}
        for name, times in self.timings.items():
            arr = np.array(times) * 1000  # Convert to ms
            result[name] = {
                "mean_ms": arr.mean(),
                "std_ms": arr.std(),
                "min_ms": arr.min(),
                "max_ms": arr.max(),
                "total_ms": arr.sum(),
                "count": len(arr),
            }
        return result

    def print_summary(self) -> None:
        summary = self.summary()
        total = sum(s["total_ms"] for s in summary.values())
        print(f"{'Phase':<20} {'Mean (ms)':>10} {'Std (ms)':>10} "
              f"{'Total (ms)':>12} {'% of Total':>10}")
        print("-" * 65)
        for name, stats in sorted(summary.items(), key=lambda x: -x[1]["total_ms"]):
            pct = stats["total_ms"] / total * 100 if total > 0 else 0
            print(f"{name:<20} {stats['mean_ms']:>10.3f} {stats['std_ms']:>10.3f} "
                  f"{stats['total_ms']:>12.1f} {pct:>9.1f}%")
        print(f"{'TOTAL':<20} {'':>10} {'':>10} {total:>12.1f} {'100.0':>9}%")


# Demo: time individual operations
set_global_seed(42)
demo_model = BenchmarkMLP().to(DEVICE)
demo_x = torch.randn(64, 100).to(DEVICE)
demo_y = torch.randint(0, 10, (64,)).to(DEVICE)
demo_loss_fn = nn.CrossEntropyLoss()
demo_opt = torch.optim.Adam(demo_model.parameters())

with SimpleTimer("forward") as t_fwd:
    out = demo_model(demo_x)
print(t_fwd)

with SimpleTimer("loss") as t_loss:
    loss = demo_loss_fn(out, demo_y)
print(t_loss)

with SimpleTimer("backward") as t_bwd:
    loss.backward()
print(t_bwd)

with SimpleTimer("optimizer.step") as t_step:
    demo_opt.step()
print(t_step)

---
## 7. Profiling a Complete Training Step

Let us break down exactly where time is spent during training.

In [ ]:
def profiled_training_epoch(
    model: nn.Module,
    train_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
    profiler: StepProfiler,
) -> float:
    """Run one training epoch with detailed timing of each phase."""
    model.train()
    total_loss = 0.0
    n_batches = 0

    for Xb, yb in train_loader:
        # Data transfer
        t0 = time.perf_counter()
        Xb, yb = Xb.to(device), yb.to(device)
        profiler.record("data_transfer", time.perf_counter() - t0)

        # Forward pass
        t0 = time.perf_counter()
        out = model(Xb)
        loss = loss_fn(out, yb)
        profiler.record("forward", time.perf_counter() - t0)

        # Backward pass
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss.backward()
        profiler.record("backward", time.perf_counter() - t0)

        # Optimizer step
        t0 = time.perf_counter()
        optimizer.step()
        profiler.record("optimizer_step", time.perf_counter() - t0)

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


# Profile 3 epochs
set_global_seed(42)
prof_model = BenchmarkMLP().to(DEVICE)
prof_opt = torch.optim.Adam(prof_model.parameters(), lr=0.001)
prof_loader = DataLoader(
    TensorDataset(X_train.to(DEVICE), y_train.to(DEVICE)),
    batch_size=64, shuffle=True,
)

profiler = StepProfiler()

for epoch in range(1, 4):
    loss = profiled_training_epoch(
        prof_model, prof_loader, prof_opt, nn.CrossEntropyLoss(), DEVICE, profiler
    )
    print(f"Epoch {epoch} | Loss: {loss:.4f}")

print("\nProfiling Summary:")
profiler.print_summary()

In [ ]:
# Visualize as a pie chart
summary = profiler.summary()
labels = list(summary.keys())
totals = [summary[k]["total_ms"] for k in labels]

fig, ax = plt.subplots(figsize=(7, 7))
colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
wedges, texts, autotexts = ax.pie(
    totals, labels=labels, autopct="%1.1f%%",
    colors=colors, startangle=90,
    textprops={"fontsize": 11},
)
ax.set_title("Time Breakdown per Training Phase")
plt.tight_layout()
plt.show()

---
## 8. Exercise: Optimize a Training Pipeline

**Task:** You are given a deliberately unoptimized training pipeline. Apply the techniques from this notebook to make it faster.

### Baseline (unoptimized)
- Batch size: 8
- No gradient accumulation
- `num_workers=0`, `pin_memory=False`
- No profiling

### Your goals
1. Profile the baseline to find bottlenecks
2. Increase batch size or use gradient accumulation for effective batch_size=128
3. Enable `pin_memory=True` (if GPU available)
4. Compare training time and final accuracy vs baseline
5. Plot both training curves

```python
# Baseline (unoptimized)
set_global_seed(42)
baseline_model = BenchmarkMLP().to(DEVICE)
baseline_opt = torch.optim.Adam(baseline_model.parameters(), lr=0.001)
baseline_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=8, shuffle=True,
    num_workers=0, pin_memory=False,
)
# TODO: Profile the baseline
# TODO: Train the optimized version
# TODO: Compare results
```

In [ ]:
# Try the exercise yourself before looking at the solution below!







### Solution

In [ ]:
# ----- Solution -----

# Step 1: Profile the baseline
print("BASELINE (bs=8, no optimizations)")
print("=" * 50)

set_global_seed(42)
baseline_model = BenchmarkMLP().to(DEVICE)
baseline_opt = torch.optim.Adam(baseline_model.parameters(), lr=0.001)
baseline_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=8, shuffle=True,
    num_workers=0, pin_memory=False,
)
val_loader_sol = DataLoader(
    TensorDataset(X_val.to(DEVICE), y_val.to(DEVICE)),
    batch_size=512,
)

baseline_profiler = StepProfiler()
baseline_history = {"train_loss": [], "val_acc": []}

t_baseline_start = time.perf_counter()
for epoch in range(1, 16):
    loss = profiled_training_epoch(
        baseline_model, baseline_loader, baseline_opt,
        nn.CrossEntropyLoss(), DEVICE, baseline_profiler,
    )
    baseline_history["train_loss"].append(loss)

    baseline_model.eval()
    correct = total = 0
    with torch.no_grad():
        for Xb, yb in val_loader_sol:
            correct += (baseline_model(Xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    baseline_history["val_acc"].append(correct / total)
t_baseline = time.perf_counter() - t_baseline_start

print(f"Baseline time: {t_baseline:.2f}s")
print(f"Baseline final val acc: {baseline_history['val_acc'][-1]:.4f}")
print("\nBaseline profile:")
baseline_profiler.print_summary()

In [ ]:
# Step 2: Optimized version
print("\nOPTIMIZED (bs=128, pin_memory=True)")
print("=" * 50)

set_global_seed(42)
opt_model = BenchmarkMLP().to(DEVICE)
# Scale LR: 0.001 * (128/8) = 0.016, but cap at 0.005 for stability
opt_optimizer = torch.optim.Adam(opt_model.parameters(), lr=0.004)
opt_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=128, shuffle=True,
    num_workers=0, pin_memory=(DEVICE.type == "cuda"),
)

opt_profiler = StepProfiler()
opt_history = {"train_loss": [], "val_acc": []}

t_opt_start = time.perf_counter()
for epoch in range(1, 16):
    loss = profiled_training_epoch(
        opt_model, opt_loader, opt_optimizer,
        nn.CrossEntropyLoss(), DEVICE, opt_profiler,
    )
    opt_history["train_loss"].append(loss)

    opt_model.eval()
    correct = total = 0
    with torch.no_grad():
        for Xb, yb in val_loader_sol:
            correct += (opt_model(Xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    opt_history["val_acc"].append(correct / total)
t_opt = time.perf_counter() - t_opt_start

print(f"Optimized time: {t_opt:.2f}s")
print(f"Optimized final val acc: {opt_history['val_acc'][-1]:.4f}")
print(f"\nSpeedup: {t_baseline / t_opt:.1f}x")
print("\nOptimized profile:")
opt_profiler.print_summary()

In [ ]:
# Step 3: Compare
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, 16)
ax1.plot(epochs, baseline_history["train_loss"], "o-", label=f"Baseline (bs=8, {t_baseline:.1f}s)", markersize=3)
ax1.plot(epochs, opt_history["train_loss"], "s--", label=f"Optimized (bs=128, {t_opt:.1f}s)", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss")
ax1.set_title("Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, baseline_history["val_acc"], "o-", label=f"Baseline (bs=8)", markersize=3)
ax2.plot(epochs, opt_history["val_acc"], "s--", label=f"Optimized (bs=128)", markersize=3)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Accuracy")
ax2.set_title("Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"  Baseline: {t_baseline:.2f}s, val_acc={baseline_history['val_acc'][-1]:.4f}")
print(f"  Optimized: {t_opt:.2f}s, val_acc={opt_history['val_acc'][-1]:.4f}")
print(f"  Speedup: {t_baseline / t_opt:.1f}x")

---
## 9. Common Mistakes & Debugging Tips

**1. Choosing batch size based only on speed**
- Larger batches are faster per epoch, but may converge to worse solutions.
- Always validate convergence quality alongside speed benchmarks.

**2. Forgetting to scale learning rate with batch size**
- Doubling batch size without adjusting LR can cause training to converge more slowly or to a worse optimum.
- Use the linear scaling rule: `new_lr = base_lr * (new_bs / base_bs)`.

**3. Gradient accumulation: forgetting to divide loss**
- Without `loss / accum_steps`, effective gradients are `accum_steps` times too large.
- This causes the optimizer to overshoot.

**4. Gradient accumulation: forgetting leftover batches**
- If `len(loader) % accum_steps != 0`, the final accumulated gradients never get stepped.
- Always handle the remainder: step after the loop if there are leftover gradients.

**5. Setting `num_workers` too high**
- More workers = more RAM. Each worker loads and stores batches independently.
- On machines with limited RAM, high `num_workers` can cause swapping or OOM.

**6. Using `num_workers > 0` on Windows without `__main__` guard**
- On Windows, multiprocessing requires `if __name__ == '__main__':` in scripts.
- In Jupyter notebooks on Windows, stick with `num_workers=0`.

**7. Not using `pin_memory` with GPU training**
- `pin_memory=True` can speed up CPU-to-GPU transfers by 2-3x.
- Combine with `tensor.to(device, non_blocking=True)` for best results.

**8. Benchmarking without warmup**
- The first few iterations are always slower (compilation, cache filling).
- Always discard the first few measurements when benchmarking.

**9. Timing GPU operations incorrectly**
- GPU operations are asynchronous. `time.perf_counter()` may not capture actual GPU time.
- For accurate GPU timing, use `torch.cuda.synchronize()` before each timing checkpoint:
  ```python
  torch.cuda.synchronize()
  t0 = time.perf_counter()
  # ... GPU operations ...
  torch.cuda.synchronize()
  elapsed = time.perf_counter() - t0
  ```

---

**Next notebook:** [04 - Reproducibility Checklist](04_Reproducibility_Checklist.ipynb) -- we cover sources of non-determinism and how to make training fully reproducible.